**Release-candidate copy.** Sanitized (private paths / account handle redacted; see REDACTION_MANIFEST.md). A guard cell was prepended so nothing runs, downloads, or calls a paid API by accident. Executed outputs (if any) are the original run outputs.

In [ ]:
# RELEASE GUARD (added to the release copy; not part of the original run)
import os
if os.environ.get("WSP_ALLOW_NOTEBOOK_RUN") != "1" or os.environ.get("WSP_PAID_API_OPT_IN") != "1":
    raise SystemExit("Historical v2 notebook: it can make PAID Anthropic API calls if a key is available. Its replies are saved in runs/v2_outputs/llm_calls.jsonl. To run anyway set WSP_ALLOW_NOTEBOOK_RUN=1 and WSP_PAID_API_OPT_IN=1.")


# WSP v2: Gradient Descent vs Weight-Space Programming, multi-task / multi-synthesizer study
* **Tasks**: modular addition (p=113), modular multiplication (p=113), sparse parity (n=16, k=6), CIFAR-10 small CNN.
* **Instance A**: gradient training (AdamW / SGD), LR sweep x seeds, LR picked on a validation split, accuracy vs steps and GPU-seconds.
* **Instance B**: weights synthesized by an LLM from a text spec (no data, no gradients), with N repeats per (task, synthesizer, prompt level).
  Synthesizers: Claude (Anthropic API) plus local open-weight models (4-bit, transformers + bitsandbytes). The prompt ablation has three levels: full hint, partial hint and no hint.
* **Instance B+ridge (CIFAR only)**: LLM-written conv filters with a closed-form ridge-regression readout. It uses data but no gradients; this is a hybrid control.
* **Instance C**: zero-gradient agentic editing (Claude) of failed or damaged B models, reporting accuracy before and after.
* A deterministic **oracle** (analytic constructor) serves as the reference for every algorithmic task.

Each section is guarded with try/except and a time limit, and results are written to `OUT_DIR` incrementally.

In [ ]:
# ---------------- Install (Kaggle: Internet ON) ----------------
import os, subprocess, sys
ON_KAGGLE = os.path.isdir("/kaggle/working")
if ON_KAGGLE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "anthropic", "bitsandbytes", "accelerate"], check=False)

In [ ]:
# ---------------- Configuration ----------------
import os, json, time, math, random, platform, re, traceback, gc, tempfile, subprocess, sys
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import pandas as pd
import matplotlib; import matplotlib.pyplot as plt

QUICK = os.environ.get("WSP_QUICK", "0") == "1"   # tiny CPU smoke test
MOCK  = os.environ.get("WSP_MOCK", "0") == "1"    # replace all LLMs by a mock synthesizer (no API / no downloads)

RUN_TASKS = {"modadd": True, "modmul": True, "parity": True, "cifar": True}
RUN_A, RUN_B, RUN_C = True, True, True

# Synthesizers. Each entry is independent and failure is non-fatal.
SYNTHESIZERS = [
    {"name": "claude-sonnet-5-5", "kind": "claude", "model_id": os.environ.get("CLAUDE_MODEL_ID", "claude-sonnet-5-5")},
    {"name": "qwen2.5-coder-14b", "kind": "hf", "model_id": "Qwen/Qwen2.5-Coder-14B-Instruct"},
    {"name": "llama-3.1-8b", "kind": "hf", "model_id": "meta-llama/Llama-3.1-8B-Instruct",
     "fallback_id": "Qwen/Qwen2.5-7B-Instruct"},   # Llama is gated: needs HF_TOKEN + accepted license
]
if MOCK: SYNTHESIZERS = [{"name": "mock", "kind": "mock", "model_id": "mock"}]

PROMPT_LEVELS = ["full", "partial", "none"]
N_REPEATS_ALGO   = 10      # Claude repeats per (task, prompt level)
N_REPEATS_LOCAL  = 3       # local-model repeats per (task, prompt level): local generation is slow on T4
N_REPEATS_CIFAR  = 3
SUCCESS_THRESH   = 0.99    # an algorithmic B run "succeeds" if all-input accuracy >= this

# Claude
CLAUDE_TEMPERATURE = 0.0   # retried without temperature if the model rejects it
CLAUDE_MAX_TOKENS  = 6000
PRICE_PER_MTOK     = {"input": 2.0, "output": 10.0}
CLAUDE_BUDGET_USD  = float(os.environ.get("CLAUDE_BUDGET_USD", "8.0"))   # HARD guard

# Local models
LOCAL_MAX_NEW_TOKENS = 1536
LOCAL_TEMPERATURE, LOCAL_TOP_P = 0.7, 0.95

# Instance A
SEEDS_A = [0, 1, 2]
LR_GRID = {"adamw": [3e-4, 1e-3, 3e-3], "sgd": [0.03, 0.1, 0.3]}
WEIGHT_DECAY = {"adamw": 1.0, "sgd": 1e-4}
CHECKPOINTS = [100, 500, 1000, 3000, 10000, 20000]
SPLIT = (0.3, 0.1)          # train, val; the rest is test
CIFAR_STEPS = [200, 1000, 3000]
CIFAR_LR_GRID = {"adam": [3e-4, 1e-3], "sgd": [0.01, 0.05]}
CIFAR_BATCH = 128

# Instance C
C_MAX_TARGETS_PER_TASK, C_ITERS = 3, 2

# Time limits per section (seconds). The whole run targets < ~6 h.
TIME_LIMIT = {"A_algo": 75*60, "A_cifar": 25*60, "B_claude": 90*60, "B_hf_each": 80*60, "C": 25*60}

if QUICK:
    SEEDS_A = [0]; LR_GRID = {"adamw": [1e-3], "sgd": [0.1]}; CHECKPOINTS = [50, 200]
    CIFAR_STEPS = [20, 50]; CIFAR_LR_GRID = {"adam": [1e-3], "sgd": [0.05]}
    N_REPEATS_ALGO = N_REPEATS_LOCAL = 2; N_REPEATS_CIFAR = 1; C_ITERS = 1

OUT_DIR = "/kaggle/working" if ON_KAGGLE else os.path.abspath(os.environ.get("WSP_OUT", "./wsp_outputs_v2"))
GEN_DIR = f"{OUT_DIR}/generated_code"; os.makedirs(GEN_DIR, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
T_START = time.time()
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
print("device", DEVICE, "| torch", torch.__version__, "| out", OUT_DIR, "| QUICK", QUICK, "| MOCK", MOCK)

## Tasks: data, shared architectures, oracle constructors, prompts

In [ ]:
P = 113
H_ALGO = 256
PAR_N, PAR_K, H_PAR = 16, 6, 64
PAR_SUBSET = sorted(np.random.default_rng(1234).choice(PAR_N, PAR_K, replace=False).tolist())

def primitive_root(p):
    fac = [q for q in range(2, p) if (p - 1) % q == 0 and all(q % r for r in range(2, int(q**.5) + 1))]
    return next(g for g in range(2, p) if all(pow(g, (p - 1) // q, p) != 1 for q in fac))
G_ROOT = primitive_root(P)

class MLP(nn.Module):
    """Shared architecture for A/B/C. Params: W1 (H,in), [b1 (H)], W2 (out,H), [b2 (out)]."""
    def __init__(self, d_in, H, d_out, act="quad", bias=False):
        super().__init__()
        self.act_name, self.bias = act, bias
        self.W1 = nn.Parameter(torch.randn(H, d_in) / math.sqrt(d_in))
        self.W2 = nn.Parameter(torch.randn(d_out, H) / math.sqrt(H))
        if bias: self.b1 = nn.Parameter(torch.zeros(H)); self.b2 = nn.Parameter(torch.zeros(d_out))
    def forward(self, x):
        h = x @ self.W1.T + (self.b1 if self.bias else 0)
        h = h * h if self.act_name == "quad" else F.relu(h)
        return h @ self.W2.T + (self.b2 if self.bias else 0)

def onehot2(a, b, p=P): return torch.cat([F.one_hot(a, p), F.one_hot(b, p)], -1).float()

def make_task(name):
    if name in ("modadd", "modmul"):
        lo = 0 if name == "modadd" else 1
        a, b = np.meshgrid(np.arange(lo, P), np.arange(lo, P), indexing="ij"); a, b = a.ravel(), b.ravel()
        y = (a + b) % P if name == "modadd" else (a * b) % P
        X = onehot2(torch.tensor(a), torch.tensor(b)); Y = torch.tensor(y)
        shapes = {"W1": (H_ALGO, 2 * P), "W2": (P, H_ALGO)}
        return dict(name=name, X=X, Y=Y, shapes=shapes, n_cls=P,
                    make_model=lambda: MLP(2 * P, H_ALGO, P, "quad", False))
    if name == "parity":
        idx = np.arange(2 ** PAR_N); bits = ((idx[:, None] >> np.arange(PAR_N)) & 1).astype(np.float32)
        y = bits[:, PAR_SUBSET].sum(1).astype(int) % 2
        shapes = {"W1": (H_PAR, PAR_N), "b1": (H_PAR,), "W2": (2, H_PAR), "b2": (2,)}
        return dict(name=name, X=torch.tensor(bits), Y=torch.tensor(y), shapes=shapes, n_cls=2,
                    make_model=lambda: MLP(PAR_N, H_PAR, 2, "relu", True))
    raise ValueError(name)

def splits(n, seed):
    perm = np.random.default_rng(seed).permutation(n); a, b = int(SPLIT[0] * n), int((SPLIT[0] + SPLIT[1]) * n)
    return torch.tensor(perm[:a]), torch.tensor(perm[a:b]), torch.tensor(perm[b:])

# ---- Oracle constructors, kept as SOURCE strings so the mock synthesizer can return them as "LLM code" ----
ORACLE_SRC = {}
ORACLE_SRC["modadd"] = r"""
def build_weights():
    p, H, M = 113, 256, 3
    K = H // (M*M); W1 = np.zeros((H, 2*p)); W2 = np.zeros((p, H)); x = np.arange(p); h = 0
    for k in range(1, K+1):
        w = 2*np.pi*k/p
        for i in range(M):
            for j in range(M):
                phi, psi = 2*np.pi*i/M, 2*np.pi*j/M
                W1[h, :p] = np.cos(w*x+phi); W1[h, p:] = np.cos(w*x+psi); W2[:, h] = np.cos(w*x+phi+psi); h += 1
    return {"W1": W1, "W2": W2 / (K*M*M/2)}
"""
ORACLE_SRC["modmul"] = r"""
def build_weights():
    p, H, M, g = 113, 256, 3, GROOT
    n = p - 1; log = np.zeros(p, dtype=int); v = 1
    for e in range(n): log[v] = e; v = (v*g) % p
    K = H // (M*M); W1 = np.zeros((H, 2*p)); W2 = np.zeros((p, H)); L = log[1:]; h = 0
    for k in range(1, K+1):
        w = 2*np.pi*k/n
        for i in range(M):
            for j in range(M):
                phi, psi = 2*np.pi*i/M, 2*np.pi*j/M
                W1[h, 1:p] = np.cos(w*L+phi); W1[h, p+1:] = np.cos(w*L+psi); W2[1:, h] = np.cos(w*L+phi+psi); h += 1
    W2[0, :] = 0.0
    return {"W1": W1, "W2": W2 / (K*M*M/2)}
""".replace("GROOT", str(G_ROOT))
ORACLE_SRC["parity"] = r"""
def build_weights():
    n, H, S = 16, 64, SUBSET
    k = len(S); W1 = np.zeros((H, n)); b1 = np.zeros(H); W2 = np.zeros((2, H)); b2 = np.array([1.0, -1.0])
    for j in range(k+1):
        W1[j, S] = 1.0; b1[j] = -j
        c = 2.0 if j == 0 else 4.0*(-1)**j
        W2[1, j] = c; W2[0, j] = -c
    return {"W1": W1, "b1": b1, "W2": W2, "b2": b2}
""".replace("SUBSET", str(PAR_SUBSET))

SYSTEM_PROMPT = ("You are an expert in mechanistic interpretability who writes neural network weights directly, "
                 "without any training, data fitting or gradients.")
FORMAT_RULES = """Return ONLY one python code block defining `def build_weights():` (no arguments) that uses only `np` (numpy, already
imported) and `math`; no other imports, no file or network access, no randomness unless seeded. It must return a dict mapping
each parameter name to a numpy array of exactly the shape listed. Do not print large arrays; compute them in code."""

def task_prompt(task, level):
    if task == "modadd":
        spec = f"""Construct weights for a network computing (a+b) mod p, p={P}, for ALL p^2 pairs a,b in 0..{P-1}.
Architecture (fixed): x = concat(onehot(a), onehot(b)) (dim {2*P}); h = W1 @ x, W1 shape ({H_ALGO}, {2*P}), no bias;
act = h*h (elementwise square); logits = W2 @ act, W2 shape ({P}, {H_ALGO}), no bias. argmax(logits) must equal (a+b) mod p."""
        hint = {"full": """Hint: Fourier features. With w_k = 2*pi*k/p, embed a and b as cos(w_k a + phase), cos(w_k b + phase'); squaring
yields cross terms cos(w_k(a+b)+phase+phase') via cos x cos y = [cos(x+y)+cos(x-y)]/2. Read out with W2[c,h] = cos(w_k c + phase+phase')
so logit_c ~ sum_k cos(w_k(a+b-c)). Use a grid of phases so unwanted terms (cos 2wa, cos 2wb, cos w(a-b), constants) cancel.""",
                "partial": "Hint: a solution exists using periodic (trigonometric) features and trig product identities.",
                "none": ""}[level]
    elif task == "modmul":
        spec = f"""Construct weights for a network computing (a*b) mod p, p={P}, for ALL pairs a,b in 1..{P-1} (0 never appears as input).
Architecture (fixed): x = concat(onehot(a), onehot(b)) with one-hot over 0..{P-1} (dim {2*P}); h = W1 @ x, W1 shape ({H_ALGO}, {2*P}), no bias;
act = h*h; logits = W2 @ act, W2 shape ({P}, {H_ALGO}), no bias, logits indexed by c in 0..{P-1}. argmax(logits) must equal (a*b) mod p."""
        hint = {"full": f"""Hint: the multiplicative group mod p is cyclic of order {P-1} with generator g={G_ROOT}. Let log(a) be the discrete log base g.
Then a*b mod p corresponds to log(a)+log(b) mod {P-1}. Apply the Fourier construction for modular addition on Z_{P-1}: embed
cos(2*pi*k*log(a)/{P-1} + phase), square, read out with cos(2*pi*k*log(c)/{P-1} + phase+phase'), phase grid so unwanted terms cancel; row c=0 can be zero.""",
                "partial": "Hint: use the structure of the multiplicative group mod p (it is cyclic) and periodic features.",
                "none": ""}[level]
    elif task == "parity":
        spec = f"""Construct weights for a network computing the parity (XOR) of the input bits at positions S={PAR_SUBSET} (0-indexed) of a
{PAR_N}-bit input x in {{0,1}}^{PAR_N}, for ALL 2^{PAR_N} inputs; the other bits are distractors.
Architecture (fixed): h = relu(W1 @ x + b1), W1 ({H_PAR}, {PAR_N}), b1 ({H_PAR},); logits = W2 @ h + b2, W2 (2, {H_PAR}), b2 (2,).
Class 1 = odd parity, class 0 = even. argmax(logits) must be correct for every input."""
        hint = {"full": f"""Hint: let s = sum of the bits in S (an integer 0..{PAR_K}). Use units relu(s - j) for j=0..{PAR_K} and choose output weights
so the piecewise-linear function of s equals +1 at odd s and -1 at even s (a zig-zag); logits = [-f, f] plus bias.""",
                "partial": "Hint: the answer only depends on the count of active bits in S; a piecewise-linear function of that count suffices.",
                "none": ""}[level]
    shapes = TASKS[task]["shapes"]
    return f"{spec}\n{hint}\nParameter names and shapes: {json.dumps({k: list(v) for k, v in shapes.items()})}\n{FORMAT_RULES}"

TASKS = {t: make_task(t) for t in ["modadd", "modmul", "parity"] if RUN_TASKS.get(t)}
for t, T in TASKS.items(): print(t, "N =", len(T["Y"]), "shapes", T["shapes"])
print("primitive root mod", P, "=", G_ROOT, "| parity subset", PAR_SUBSET)

## Sandboxed execution of generated code, evaluation, structure metrics

In [ ]:
RUNNER = r"""
import sys, json, math, numpy as np, builtins as _b
src = open(sys.argv[1]).read().replace("import numpy as np", "").replace("import math", "")
safe = {k: getattr(_b, k) for k in ["range","len","int","float","abs","min","max","sum","enumerate","zip","list","dict","tuple",
        "round","isinstance","ValueError","print","reversed","sorted","map","pow","bool","set","any","all","divmod","Exception","str"]}
ns = {"np": np, "math": math, "__builtins__": safe}
exec(src, ns)
fn = ns[sys.argv[3]]
W = fn() if sys.argv[3] == "build_weights" else fn(dict(np.load(sys.argv[4])))
np.savez(sys.argv[2], **{k: np.asarray(v, dtype=np.float64) for k, v in W.items()})
"""
BANNED = ["import os", "import sys", "subprocess", "open(", "__import__", "eval(", "exec(", "socket", "torch", "import ", "globals(", "getattr("]

def extract_code(text):
    m = re.findall(r"```(?:python|py)?\s*\n(.*?)```", text, re.S)
    if m: return max(m, key=len)
    return text

def run_generated(code_str, fn_name="build_weights", inputs=None, timeout=120):
    """Run generated code in a SUBPROCESS (timeout, restricted builtins, numpy only). Not a full sandbox: review logged code."""
    clean = code_str.replace("import numpy as np", "").replace("import math", "")
    for bw in BANNED:
        if bw in clean: raise ValueError(f"disallowed token in generated code: {bw!r}")
    d = tempfile.mkdtemp(); src, out, inp, run = f"{d}/g.py", f"{d}/w.npz", f"{d}/in.npz", f"{d}/runner.py"
    open(src, "w").write(clean); open(run, "w").write(RUNNER)
    np.savez(inp, **(inputs or {"_": np.zeros(1)}))
    r = subprocess.run([sys.executable, run, src, out, fn_name, inp], capture_output=True, text=True, timeout=timeout)
    if r.returncode != 0: raise RuntimeError("generated code failed: " + r.stderr[-600:])
    return {k: v for k, v in np.load(out).items()}

def validate(W, shapes):
    assert isinstance(W, dict), "weights must be a dict"
    out = {}
    for k, shp in shapes.items():
        assert k in W, f"missing parameter {k}"
        a = np.asarray(W[k], dtype=np.float64)
        if a.shape != tuple(shp) and a.size == int(np.prod(shp)): a = a.reshape(shp)
        assert a.shape == tuple(shp), f"{k} shape {a.shape} != {tuple(shp)}"
        assert np.isfinite(a).all(), f"{k} has non-finite values"
        out[k] = a
    return out

def load_weights(model, W):
    sd = model.state_dict()
    with torch.no_grad():
        for k in sd: sd[k].copy_(torch.tensor(W[k], dtype=sd[k].dtype))
    return model.to(DEVICE)

def get_weights(model): return {k: v.detach().cpu().numpy().astype(np.float64) for k, v in model.state_dict().items()}

@torch.no_grad()
def evaluate(model, X, Y, idx=None, noise=0.0, seed=0, bs=16384):
    model.eval().to(DEVICE)
    if idx is not None: X, Y = X[idx], Y[idx]
    g = torch.Generator().manual_seed(seed); preds, margins, losses = [], [], []
    for i in range(0, len(Y), bs):
        x = X[i:i+bs]
        if noise > 0: x = x + noise * torch.randn(x.shape, generator=g)
        x, y = x.to(DEVICE), Y[i:i+bs].to(DEVICE)
        lg = model(x).float()
        lg = torch.nan_to_num(lg, nan=0.0, posinf=1e30, neginf=-1e30)
        preds.append(lg.argmax(-1).cpu())
        c = lg.gather(1, y[:, None]).squeeze(1); o = lg.clone(); o.scatter_(1, y[:, None], -float("inf"))
        margins.append((c - o.max(-1).values).cpu()); losses.append(F.cross_entropy(lg, y, reduction="sum").item())
    pred, m = torch.cat(preds), torch.cat(margins)
    wrong = (pred != Y.cpu()).nonzero().squeeze(1)
    return {"acc": (pred == Y.cpu()).float().mean().item(), "loss": sum(losses) / len(Y),
            "mean_margin": m.mean().item(), "min_margin": m.min().item(), "wrong": wrong[:2000].tolist(), "pred": pred}

def fourier_energy(task, W):
    """Fraction of W1 embedding power in its top-10 frequencies (Fourier over a, or over log(a) for modmul)."""
    if task not in ("modadd", "modmul") or "W1" not in W: return float("nan")
    E = W["W1"][:, :P]
    if task == "modmul":
        order = [pow(G_ROOT, e, P) for e in range(P - 1)]; E = E[:, order]
    pw = (np.abs(np.fft.rfft(E, axis=1)) ** 2).sum(0)[1:]
    return float(np.sort(pw)[::-1][:10].sum() / (pw.sum() + 1e-12))

def structure(task, W):
    out = {}
    for name, A in W.items():
        if A.ndim < 2: continue
        A2 = A.reshape(A.shape[0], -1); s = np.linalg.svd(A2, compute_uv=False)
        if s[0] <= 0: out.update({f"{name}_rank": 0, f"{name}_eff_rank": 0.0, f"{name}_sparsity": 1.0}); continue
        q = s / s.sum()
        out[f"{name}_rank"] = int((s > 1e-3 * s[0]).sum()); out[f"{name}_eff_rank"] = float(np.exp(-(q @ np.log(q + 1e-12))))
        out[f"{name}_sparsity"] = float((np.abs(A2) < 1e-3 * np.abs(A2).max()).mean()); out[f"{name}_fro"] = float(np.linalg.norm(A2))
    out["fourier_top10_energy"] = fourier_energy(task, W)
    return out

NOISES = [0.1, 0.3, 0.5]
def full_metrics(task, model, T, sp=None):
    r_all = evaluate(model, T["X"], T["Y"])
    row = {"all_acc": r_all["acc"], "mean_margin": r_all["mean_margin"], "min_margin": r_all["min_margin"]}
    if sp is not None: row["test_acc"] = evaluate(model, T["X"], T["Y"], sp[2])["acc"]
    for nz in NOISES: row[f"acc_noise{nz}"] = evaluate(model, T["X"], T["Y"], noise=nz)["acc"]
    row.update(structure(task, get_weights(model))); return row

# ---- incremental result store ----
PER_RUN, A_CURVES, C_ROWS = [], [], []
def save_all():
    pd.DataFrame(PER_RUN).to_csv(f"{OUT_DIR}/per_run.csv", index=False)
    pd.DataFrame(A_CURVES).to_csv(f"{OUT_DIR}/A_curves.csv", index=False)
    pd.DataFrame(C_ROWS).to_csv(f"{OUT_DIR}/C_edits.csv", index=False)
    json.dump({"per_run": PER_RUN, "C": C_ROWS, "llm_usage": USAGE if "USAGE" in globals() else {}, "elapsed_s": time.time() - T_START},
              open(f"{OUT_DIR}/results_partial.json", "w"), default=str)

# Sanity: every oracle must be 100%
ORACLE_W = {}
for t, T in TASKS.items():
    W = validate(run_generated(ORACLE_SRC[t]), T["shapes"]); ORACLE_W[t] = W
    m = load_weights(T["make_model"](), W); r = full_metrics(t, m, T)
    PER_RUN.append({"task": t, "instance": "B_oracle", "synth": "oracle", "prompt": "-", "rep": 0, "ok": True, **r})
    print(f"oracle {t}: acc {r['all_acc']:.4f} min margin {r['min_margin']:.3f}")
save_all()

## Instance A: gradient baselines (LR sweep x optimizer x seeds; LR selected on validation)

In [ ]:
def train_A_algo(task, T, opt_name, lr, seed, t_deadline):
    set_seed(seed); sp = splits(len(T["Y"]), seed); m = T["make_model"]().to(DEVICE)
    if opt_name == "adamw": opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=WEIGHT_DECAY["adamw"], betas=(0.9, 0.98))
    else: opt = torch.optim.SGD(m.parameters(), lr=lr, momentum=0.9, weight_decay=WEIGHT_DECAY["sgd"])
    xtr, ytr = T["X"][sp[0]].to(DEVICE), T["Y"][sp[0]].to(DEVICE); gpu_s = 0.0; rows = []
    for step in range(1, max(CHECKPOINTS) + 1):
        t0 = time.perf_counter(); m.train()
        loss = F.cross_entropy(m(xtr), ytr); opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
        if DEVICE == "cuda": torch.cuda.synchronize()
        gpu_s += time.perf_counter() - t0
        if step in CHECKPOINTS:
            rows.append({"task": task, "opt": opt_name, "lr": lr, "seed": seed, "step": step, "train_loss": loss.item(),
                         "val_acc": evaluate(m, T["X"], T["Y"], sp[1])["acc"], "test_acc": evaluate(m, T["X"], T["Y"], sp[2])["acc"],
                         "train_compute_s": gpu_s})
        if time.time() > t_deadline: break
    return m, rows, sp

def run_A_algo():
    deadline = time.time() + TIME_LIMIT["A_algo"]
    for t, T in TASKS.items():
        for opt_name, lrs in LR_GRID.items():
            best = None
            for lr in lrs:
                for seed in SEEDS_A:
                    if time.time() > deadline: print("A_algo time limit hit"); return
                    try:
                        m, rows, sp = train_A_algo(t, T, opt_name, lr, seed, deadline)
                        A_CURVES.extend(rows); last = rows[-1]
                        r = full_metrics(t, m, T, sp)
                        PER_RUN.append({"task": t, "instance": "A", "synth": f"{opt_name}", "prompt": "-", "rep": seed, "lr": lr,
                                        "steps": last["step"], "val_acc": last["val_acc"], "compute_s": last["train_compute_s"], "ok": True, **r})
                        print(f"[A {t} {opt_name} lr={lr} s={seed}] val {last['val_acc']:.3f} test {r['test_acc']:.3f} ({last['train_compute_s']:.0f}s)")
                    except Exception as e:
                        print("A run failed:", t, opt_name, lr, seed, repr(e)); traceback.print_exc()
                    save_all()
if RUN_A:
    try: run_A_algo()
    except Exception as e: print("A_algo section failed:", repr(e))

## LLM synthesizers: Claude (budget-guarded) and local HF 4-bit models

In [ ]:
LLM_LOG = f"{OUT_DIR}/llm_calls.jsonl"
USAGE = {}            # name -> {"calls", "input", "output", "cost_usd", "latency_s", "errors"}
CLAUDE_SPENT = [0.0]
class BudgetExceeded(Exception): pass

def _secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name)

def _log(rec):
    with open(LLM_LOG, "a") as f: f.write(json.dumps(rec, default=str) + "\n")

def _u(name):
    return USAGE.setdefault(name, {"calls": 0, "input": 0, "output": 0, "cost_usd": 0.0, "latency_s": 0.0, "errors": 0})

_CLAUDE = {}
def call_claude(syn, system, user, tag):
    worst = (len(user) / 3.5 * PRICE_PER_MTOK["input"] + CLAUDE_MAX_TOKENS * PRICE_PER_MTOK["output"]) / 1e6
    if CLAUDE_SPENT[0] + worst > CLAUDE_BUDGET_USD:
        _log({"tag": tag, "synth": syn["name"], "event": "BUDGET_STOP", "spent": CLAUDE_SPENT[0], "budget": CLAUDE_BUDGET_USD})
        raise BudgetExceeded(f"Claude budget guard: spent ${CLAUDE_SPENT[0]:.3f}, next call could exceed ${CLAUDE_BUDGET_USD}")
    if "client" not in _CLAUDE:
        key = _secret("ANTHROPIC_API_KEY")
        if not key: raise RuntimeError("No ANTHROPIC_API_KEY secret")
        import anthropic; _CLAUDE["client"] = anthropic.Anthropic(api_key=key)
    client = _CLAUDE["client"]
    kw = dict(model=syn["model_id"], max_tokens=CLAUDE_MAX_TOKENS, system=system, messages=[{"role": "user", "content": user}])
    t0 = time.perf_counter(); temp = CLAUDE_TEMPERATURE
    if _CLAUDE.get("no_temp"): temp = None; resp = client.messages.create(**kw)
    else:
        try: resp = client.messages.create(temperature=CLAUDE_TEMPERATURE, **kw)
        except Exception as e:
            if "temperature" not in str(e).lower(): raise
            _CLAUDE["no_temp"] = True; temp = None; resp = client.messages.create(**kw)
    dt = time.perf_counter() - t0
    text = "".join(b.text for b in resp.content if getattr(b, "type", "") == "text")
    ui, uo = resp.usage.input_tokens, resp.usage.output_tokens
    cost = (ui * PRICE_PER_MTOK["input"] + uo * PRICE_PER_MTOK["output"]) / 1e6; CLAUDE_SPENT[0] += cost
    u = _u(syn["name"]); u["calls"] += 1; u["input"] += ui; u["output"] += uo; u["cost_usd"] += cost; u["latency_s"] += dt
    _log({"tag": tag, "synth": syn["name"], "model_requested": syn["model_id"], "model_returned": resp.model, "temperature": temp,
          "system": system, "user": user, "response": text, "usage": {"input": ui, "output": uo}, "cost_usd": cost,
          "cum_cost_usd": CLAUDE_SPENT[0], "latency_s": dt, "stop_reason": resp.stop_reason, "time": time.strftime("%Y-%m-%dT%H:%M:%S%z")})
    return text, {"in_tok": ui, "out_tok": uo, "cost_usd": cost, "latency_s": dt}

_HF = {}
def load_hf(syn):
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    tok_kw = {"token": _secret("HF_TOKEN")} if _secret("HF_TOKEN") else {}
    ids = [syn["model_id"]] + ([syn["fallback_id"]] if syn.get("fallback_id") else [])
    last = None
    for mid in ids:
        try:
            q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
            tok = AutoTokenizer.from_pretrained(mid, **tok_kw)
            mdl = AutoModelForCausalLM.from_pretrained(mid, quantization_config=q, device_map="auto", torch_dtype=torch.float16, **tok_kw)
            mdl.eval(); _HF.update(tok=tok, model=mdl, id=mid)
            if mid != syn["model_id"]: print(f"NOTE: {syn['model_id']} unavailable -> using fallback {mid}")
            syn["resolved_id"] = mid; return
        except Exception as e:
            last = e; print(f"could not load {mid}: {repr(e)[:300]}")
    raise RuntimeError(f"no loadable model for {syn['name']}: {last!r}")

def unload_hf():
    _HF.clear(); gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

@torch.no_grad()
def call_hf(syn, system, user, tag):
    tok, mdl = _HF["tok"], _HF["model"]
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt")
    if not torch.is_tensor(ids): ids = ids["input_ids"]       # newer transformers return a BatchEncoding
    ids = ids.to(mdl.device)
    t0 = time.perf_counter()
    out = mdl.generate(ids, max_new_tokens=LOCAL_MAX_NEW_TOKENS, do_sample=True, temperature=LOCAL_TEMPERATURE, top_p=LOCAL_TOP_P,
                       pad_token_id=tok.eos_token_id)
    dt = time.perf_counter() - t0; gen = out[0, ids.shape[1]:]; text = tok.decode(gen, skip_special_tokens=True)
    u = _u(syn["name"]); u["calls"] += 1; u["input"] += ids.shape[1]; u["output"] += len(gen); u["latency_s"] += dt
    _log({"tag": tag, "synth": syn["name"], "model_requested": syn["model_id"], "model_returned": _HF["id"],
          "temperature": LOCAL_TEMPERATURE, "top_p": LOCAL_TOP_P, "system": system, "user": user, "response": text,
          "usage": {"input": int(ids.shape[1]), "output": int(len(gen))}, "latency_s": dt, "time": time.strftime("%Y-%m-%dT%H:%M:%S%z")})
    return text, {"in_tok": int(ids.shape[1]), "out_tok": int(len(gen)), "cost_usd": 0.0, "latency_s": dt}

_MOCK_RNG = np.random.default_rng(0)
def call_mock(syn, system, user, tag):
    """Offline stand-in for testing the pipeline: returns the oracle for full hints, garbage/partial code otherwise."""
    task = next((t for t in ORACLE_SRC if f"[[task={t}]]" in user), None)
    r = _MOCK_RNG.random()
    if tag.startswith("C_"):
        text = "```python\ndef edit_weights(W):\n    W = dict(W)\n    W['W2'] = W['W2'] * 1.0\n    return W\n```"
    elif task and ("Hint: Fourier" in user or "Hint: the multiplicative" in user or "Hint: let s" in user) and r < 0.8:
        text = "```python\n" + ORACLE_SRC[task] + "```"
    elif r < 0.5:
        text = "```python\ndef build_weights():\n    return {'W1': np.zeros((3,3))}\n```"   # wrong shapes -> failure path
    else:
        text = "no code here"
    u = _u(syn["name"]); u["calls"] += 1
    _log({"tag": tag, "synth": "mock", "user": user[:200], "response": text})
    return text, {"in_tok": 0, "out_tok": 0, "cost_usd": 0.0, "latency_s": 0.0}

def call_llm(syn, system, user, tag):
    k = syn["kind"]
    try:
        return {"claude": call_claude, "hf": call_hf, "mock": call_mock}[k](syn, system, user, tag)
    except BudgetExceeded: raise
    except Exception:
        _u(syn["name"])["errors"] += 1; raise

## Instance B: synthesis on algorithmic tasks (repeats x prompt ablation x synthesizer)

In [ ]:
def synth_once(syn, task, level, rep):
    T = TASKS[task]; user = task_prompt(task, level) + (f"\n[[task={task}]]" if syn["kind"] == "mock" else "")
    tag = f"B_{task}_{level}_r{rep}"; t0 = time.perf_counter()
    row = {"task": task, "instance": "B", "synth": syn["name"], "synth_model": syn.get("resolved_id", syn["model_id"]),
           "prompt": level, "rep": rep, "ok": False, "all_acc": 0.0, "error": None}
    try:
        text, meta = call_llm(syn, SYSTEM_PROMPT, user, tag); row.update(meta)
        code_str = extract_code(text)
        fn = f"{GEN_DIR}/{syn['name']}_{task}_{level}_r{rep}.py"; open(fn, "w").write(code_str)
        W = validate(run_generated(code_str), T["shapes"])
        m = load_weights(T["make_model"](), W); row.update(full_metrics(task, m, T)); row["ok"] = True
        np.savez_compressed(f"{GEN_DIR}/{syn['name']}_{task}_{level}_r{rep}_weights.npz", **W)
    except BudgetExceeded: raise
    except Exception as e:
        row["error"] = repr(e)[:400]
    row["synth_wall_s"] = time.perf_counter() - t0
    row["success"] = bool(row["ok"] and row["all_acc"] >= SUCCESS_THRESH)
    return row

def run_B_algo(syn):
    reps = N_REPEATS_ALGO if syn["kind"] in ("claude", "mock") else N_REPEATS_LOCAL
    limit = TIME_LIMIT["B_claude"] if syn["kind"] != "hf" else TIME_LIMIT["B_hf_each"]
    deadline = time.time() + limit
    for rep in range(reps):                      # rep-major order: partial results stay balanced across cells
        for task in TASKS:
            for level in PROMPT_LEVELS:
                if time.time() > deadline: print(f"[{syn['name']}] time limit hit"); return "timeout"
                row = synth_once(syn, task, level, rep); PER_RUN.append(row); save_all()
                print(f"[B {syn['name']} {task} {level} r{rep}] ok={row['ok']} acc={row['all_acc']:.4f} {row['error'] or ''}"[:220])
    return "done"

## CIFAR-10: A (SGD/Adam at fixed step budgets) vs B (LLM-written conv filters + head) vs B+ridge

In [ ]:
CIFAR_SHAPES = {"conv1.weight": (32, 3, 5, 5), "conv1.bias": (32,), "conv2.weight": (64, 32, 3, 3), "conv2.bias": (64,),
                "fc.weight": (10, 1024), "fc.bias": (10,)}
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 5, padding=2); self.conv2 = nn.Conv2d(32, 64, 3, padding=1); self.fc = nn.Linear(1024, 10)
    def features(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)          # 32x16x16
        x = F.adaptive_avg_pool2d(F.relu(self.conv2(x)), 4)  # 64x4x4
        return x.flatten(1)
    def forward(self, x): return self.fc(self.features(x))

CIFAR_CLASSES = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
CIFAR_MEAN, CIFAR_STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)

def load_cifar():
    import torchvision, torchvision.transforms as tf
    root = "/kaggle/temp/cifar" if ON_KAGGLE else "./cifar_data"
    try:
        if QUICK or MOCK: raise RuntimeError("offline/quick mode")
        tr = torchvision.datasets.CIFAR10(root, train=True, download=True); te = torchvision.datasets.CIFAR10(root, train=False, download=True)
        to = lambda d: (torch.tensor(d.data).permute(0, 3, 1, 2).float() / 255., torch.tensor(d.targets))
        (xtr, ytr), (xte, yte) = to(tr), to(te); src = "CIFAR-10"
    except Exception as e:
        print("CIFAR-10 unavailable -> synthetic stand-in (pipeline test only):", repr(e)[:120])
        g = torch.Generator().manual_seed(0); xtr = torch.rand(2000, 3, 32, 32, generator=g); ytr = torch.randint(0, 10, (2000,), generator=g)
        xte = torch.rand(500, 3, 32, 32, generator=g); yte = torch.randint(0, 10, (500,), generator=g); src = "FAKE"
    m = torch.tensor(CIFAR_MEAN)[None, :, None, None]; s = torch.tensor(CIFAR_STD)[None, :, None, None]
    return (xtr - m) / s, ytr, (xte - m) / s, yte, src

CIFAR_PROMPT = f"""Write the weights of a small CNN for CIFAR-10 classification WITHOUT any training, data, or gradients.
Input: 3x32x32 RGB images normalized per channel with mean {CIFAR_MEAN} and std {CIFAR_STD}. Classes (index order): {CIFAR_CLASSES}.
Architecture (PyTorch, fixed): conv1 = Conv2d(3,32,5,padding=2) -> ReLU -> MaxPool2d(2) -> conv2 = Conv2d(32,64,3,padding=1) -> ReLU
-> AdaptiveAvgPool2d(4) -> flatten (1024 = 64 channels x 4 x 4, channel-major) -> fc = Linear(1024,10).
Design meaningful filters from first principles (e.g. oriented edge/Gabor, color-opponent, center-surround, blob detectors in conv1;
combinations/texture detectors in conv2) and a classifier head encoding your prior knowledge of which colors, textures and spatial
layouts indicate each class (e.g. blue/uniform top regions for airplane/ship, green/brown textures for frog/deer, etc.).
Parameter names and shapes: {json.dumps({k: list(v) for k, v in CIFAR_SHAPES.items()})}
{FORMAT_RULES}"""

@torch.no_grad()
def eval_cifar(m, X, Y, noise=0.0, bs=1000):
    m.eval().to(DEVICE); c = 0; g = torch.Generator().manual_seed(0)
    for i in range(0, len(Y), bs):
        x = X[i:i+bs]
        if noise > 0: x = x + noise * torch.randn(x.shape, generator=g)
        lg = torch.nan_to_num(m(x.to(DEVICE)).float()); c += (lg.argmax(-1).cpu() == Y[i:i+bs]).sum().item()
    return c / len(Y)

def cifar_metrics(m, Xte, Yte):
    r = {"all_acc": eval_cifar(m, Xte, Yte), "test_acc": None}
    r["test_acc"] = r["all_acc"]
    for nz in NOISES: r[f"acc_noise{nz}"] = eval_cifar(m, Xte, Yte, nz)
    r.update(structure("cifar", get_weights(m))); return r

def train_A_cifar(Xtr, Ytr, Xval, Yval, opt_name, lr, seed, deadline):
    set_seed(seed); m = SmallCNN().to(DEVICE)
    opt = torch.optim.Adam(m.parameters(), lr=lr) if opt_name == "adam" else torch.optim.SGD(m.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    g = torch.Generator().manual_seed(seed); comp = 0.0; rows = []
    for step in range(1, max(CIFAR_STEPS) + 1):
        t0 = time.perf_counter(); idx = torch.randint(0, len(Ytr), (CIFAR_BATCH,), generator=g)
        x, y = Xtr[idx].to(DEVICE), Ytr[idx].to(DEVICE)
        if torch.rand(1, generator=g).item() < 0.5: x = x.flip(3)
        m.train(); loss = F.cross_entropy(m(x), y); opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        if DEVICE == "cuda": torch.cuda.synchronize()
        comp += time.perf_counter() - t0
        if step in CIFAR_STEPS:
            rows.append({"task": "cifar", "opt": opt_name, "lr": lr, "seed": seed, "step": step, "train_loss": loss.item(),
                         "val_acc": eval_cifar(m, Xval, Yval), "train_compute_s": comp})
        if time.time() > deadline: break
    return m, rows

def ridge_head(m, Xtr, Ytr, lam=1.0, n=10000):
    """Closed-form ridge readout on frozen (LLM-written) conv features. Uses data, no gradients."""
    with torch.no_grad():
        Fs = torch.cat([m.features(Xtr[i:i+1000].to(DEVICE)).cpu() for i in range(0, min(n, len(Ytr)), 1000)]).double()
        Y1 = F.one_hot(Ytr[:len(Fs)], 10).double() * 2 - 1
        mu, sd = Fs.mean(0), Fs.std(0) + 1e-6; Z = torch.cat([(Fs - mu) / sd, torch.ones(len(Fs), 1, dtype=torch.float64)], 1)
        Wr = torch.linalg.solve(Z.T @ Z + lam * len(Fs) * torch.eye(Z.shape[1], dtype=torch.float64), Z.T @ Y1)   # (1025,10)
        Wf = (Wr[:-1] / sd[:, None]).T; bf = Wr[-1] - (mu / sd) @ Wr[:-1]
        m2 = SmallCNN(); m2.load_state_dict(m.state_dict())
        m2.fc.weight.copy_(Wf.float()); m2.fc.bias.copy_(bf.float())
    return m2.to(DEVICE)

CIFAR_DATA = None
def run_cifar_A():
    global CIFAR_DATA
    Xtr, Ytr, Xte, Yte, src = CIFAR_DATA
    nval = min(5000, len(Ytr) // 10); Xval, Yval, Xt, Yt = Xtr[-nval:], Ytr[-nval:], Xtr[:-nval], Ytr[:-nval]
    deadline = time.time() + TIME_LIMIT["A_cifar"]
    for opt_name, lrs in CIFAR_LR_GRID.items():
        for lr in lrs:
            for seed in SEEDS_A:
                if time.time() > deadline: print("A_cifar time limit"); return
                try:
                    m, rows = train_A_cifar(Xt, Yt, Xval, Yval, opt_name, lr, seed, deadline); A_CURVES.extend(rows); last = rows[-1]
                    r = cifar_metrics(m, Xte, Yte)
                    PER_RUN.append({"task": "cifar", "instance": "A", "synth": opt_name, "prompt": "-", "rep": seed, "lr": lr, "steps": last["step"],
                                    "val_acc": last["val_acc"], "compute_s": last["train_compute_s"], "ok": True, "data": src, **r})
                    print(f"[A cifar {opt_name} lr={lr} s={seed}] val {last['val_acc']:.3f} test {r['test_acc']:.3f}")
                except Exception as e: print("A cifar run failed", repr(e))
                save_all()

def run_cifar_B(syn):
    Xtr, Ytr, Xte, Yte, src = CIFAR_DATA
    for rep in range(N_REPEATS_CIFAR):
        row = {"task": "cifar", "instance": "B", "synth": syn["name"], "synth_model": syn.get("resolved_id", syn["model_id"]),
               "prompt": "full", "rep": rep, "ok": False, "all_acc": 0.0, "error": None, "data": src}
        t0 = time.perf_counter()
        try:
            if syn["kind"] == "mock":
                code_str = "def build_weights():\n    r = np.random.default_rng(0)\n    return {k: r.standard_normal(s)*0.1 for k, s in " + repr(CIFAR_SHAPES) + ".items()}\n"
                meta = {}
            else:
                text, meta = call_llm(syn, SYSTEM_PROMPT, CIFAR_PROMPT, f"B_cifar_r{rep}"); code_str = extract_code(text)
            row.update(meta); open(f"{GEN_DIR}/{syn['name']}_cifar_r{rep}.py", "w").write(code_str)
            W = validate(run_generated(code_str, timeout=300), CIFAR_SHAPES)
            m = load_weights(SmallCNN(), W); row.update(cifar_metrics(m, Xte, Yte)); row["ok"] = True
            mr = ridge_head(m, Xtr, Ytr); rr = cifar_metrics(mr, Xte, Yte)
            PER_RUN.append({**row, "instance": "B+ridge", **rr, "synth_wall_s": time.perf_counter() - t0})
        except BudgetExceeded: raise
        except Exception as e: row["error"] = repr(e)[:400]
        row["synth_wall_s"] = time.perf_counter() - t0; PER_RUN.append(row); save_all()
        print(f"[B cifar {syn['name']} r{rep}] ok={row['ok']} acc={row['all_acc']:.4f} {row['error'] or ''}"[:220])

if RUN_TASKS.get("cifar"):
    try:
        CIFAR_DATA = load_cifar(); print("CIFAR source:", CIFAR_DATA[4], CIFAR_DATA[0].shape)
        if RUN_A: run_cifar_A()
    except Exception as e: print("CIFAR A section failed:", repr(e)); traceback.print_exc()

## Run all synthesizers (one model loaded at a time; failures are non-fatal)

In [ ]:
SYN_STATUS = {}
if RUN_B:
    for syn in SYNTHESIZERS:
        status = []
        try:
            if syn["kind"] == "hf": load_hf(syn)
            if TASKS: status.append("algo:" + run_B_algo(syn))
            if CIFAR_DATA is not None: run_cifar_B(syn); status.append("cifar:done")
        except BudgetExceeded as e: status.append("BUDGET_STOP"); print(e)
        except Exception as e: status.append("FAILED " + repr(e)[:200]); print(f"synthesizer {syn['name']} failed:", repr(e)); traceback.print_exc()
        finally:
            if syn["kind"] == "hf": unload_hf()
        SYN_STATUS[syn["name"]] = {"status": status, "resolved_id": syn.get("resolved_id", syn["model_id"])}; save_all()
print(json.dumps(SYN_STATUS, indent=1)); print("usage", json.dumps(USAGE, indent=1), "| Claude spend $%.3f" % CLAUDE_SPENT[0])

## Instance C: zero-gradient agentic editing (Claude)
Targets per algorithmic task are up to `C_MAX_TARGETS_PER_TASK` failed-but-runnable Claude B models, plus one **damaged oracle** (80% of hidden units zeroed and N(0,0.5) noise on W1) so the repair is measurable even when B is perfect. The editor sees an error report and returns `edit_weights(W)`. Each iteration is evaluated, and the best result is kept.

In [ ]:
def error_report(task, model, T):
    r = evaluate(model, T["X"], T["Y"]); W = get_weights(model); wrong = r["wrong"]
    X, Y = T["X"], T["Y"]
    rep = {"acc": r["acc"], "n_wrong": int((r["pred"] != Y).sum()), "min_margin": r["min_margin"], "mean_margin": r["mean_margin"],
           "dead_hidden_units": int((np.abs(W["W1"]).sum(1) < 1e-8).sum()),
           "param_norms": {k: float(np.linalg.norm(v)) for k, v in W.items()}}
    if task in ("modadd", "modmul"):
        ab = [(int(X[i, :P].argmax()), int(X[i, P:].argmax()), int(Y[i]), int(r["pred"][i])) for i in wrong[:20]]
        rep["sample_wrong (a,b,true,pred)"] = ab; rep["fourier_top10_energy"] = fourier_energy(task, W)
    else:
        rep["sample_wrong (bits,true,pred)"] = [("".join(str(int(v)) for v in X[i].tolist()), int(Y[i]), int(r["pred"][i])) for i in wrong[:20]]
    return rep

def damage_weights(W, seed=0):
    g = np.random.default_rng(seed); W = {k: v.copy() for k, v in W.items()}; H = W["W1"].shape[0]
    dead = g.choice(H, int(0.8 * H), replace=False); W["W1"][dead] = 0; W["W2"][:, dead] = 0
    if "b1" in W: W["b1"][dead] = 0
    W["W1"] += 0.5 * g.standard_normal(W["W1"].shape); return W

C_PROMPT = """You are repairing the weights of the network below WITHOUT gradients or training.
{spec}
The current weights are available to your code as a dict W of numpy arrays (same names/shapes as above).
Error report of the current weights (JSON): {report}
Return ONLY one python code block defining `def edit_weights(W):` that returns a dict with ALL parameters (same names and shapes).
Use only np and math; no imports. You may rewrite any part of the weights."""

def run_C():
    claude = next((s for s in SYNTHESIZERS if s["kind"] in ("claude", "mock")), None)
    if claude is None: print("No Claude synthesizer configured -> skip C"); return
    deadline = time.time() + TIME_LIMIT["C"]
    for task, T in TASKS.items():
        fails = [r for r in PER_RUN if r.get("instance") == "B" and r["task"] == task and r["synth"] == claude["name"]
                 and r["ok"] and not r["success"]][:C_MAX_TARGETS_PER_TASK]
        targets = [("damaged_oracle", damage_weights(ORACLE_W[task]))]
        for r in fails:
            f = f"{GEN_DIR}/{claude['name']}_{task}_{r['prompt']}_r{r['rep']}_weights.npz"
            if os.path.exists(f): targets.append((f"B_{r['prompt']}_r{r['rep']}", dict(np.load(f))))
        for name, W0 in targets:
            if time.time() > deadline: print("C time limit"); return
            cur = W0; m = load_weights(T["make_model"](), cur); acc0 = evaluate(m, T["X"], T["Y"])["acc"]; best = (acc0, cur)
            row = {"task": task, "target": name, "acc_before": acc0, "iters": []}
            for it in range(C_ITERS):
                try:
                    rep = error_report(task, m, T)
                    text, meta = call_llm(claude, SYSTEM_PROMPT, C_PROMPT.format(spec=task_prompt(task, "full"), report=json.dumps(rep, default=str)),
                                          f"C_{task}_{name}_it{it}")
                    code_str = extract_code(text); open(f"{GEN_DIR}/C_{task}_{name}_it{it}.py", "w").write(code_str)
                    cur = validate(run_generated(code_str, "edit_weights", inputs=cur), T["shapes"])
                    m = load_weights(T["make_model"](), cur); a = evaluate(m, T["X"], T["Y"])["acc"]
                    row["iters"].append({"it": it, "acc": a, **meta})
                    if a > best[0]: best = (a, cur)
                    if a >= 0.9999: break
                except BudgetExceeded as e: row["iters"].append({"it": it, "error": "BUDGET_STOP"}); break
                except Exception as e: row["iters"].append({"it": it, "error": repr(e)[:300]})
            row["acc_after"] = best[0]; C_ROWS.append(row); save_all()
            print(f"[C {task} {name}] {acc0:.4f} -> {best[0]:.4f}")
if RUN_C:
    try: run_C()
    except Exception as e: print("C section failed:", repr(e)); traceback.print_exc()

## Aggregation: mean, std, 95% CI, success rate (Wilson CI)

In [ ]:
def t_ci(x):
    x = np.asarray([v for v in x if v is not None and np.isfinite(v)], float); n = len(x)
    if n == 0: return np.nan, np.nan, np.nan
    if n == 1: return x[0], 0.0, 0.0
    try:
        from scipy import stats; tq = stats.t.ppf(0.975, n - 1)
    except Exception: tq = 1.96
    return x.mean(), x.std(ddof=1), tq * x.std(ddof=1) / math.sqrt(n)

def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    ph = k / n; d = 1 + z * z / n; c = (ph + z * z / (2 * n)) / d; h = z * math.sqrt(ph * (1 - ph) / n + z * z / (4 * n * n)) / d
    return (c - h, c + h)

df = pd.DataFrame(PER_RUN)
# For A, keep only the LR chosen on validation (per task, optimizer), averaged over seeds.
sel = []
if len(df) and "lr" in df:
    for (t, s), g in df[df.instance == "A"].groupby(["task", "synth"]):
        best_lr = g.groupby("lr")["val_acc"].mean().idxmax(); sel.append(g[g.lr == best_lr])
dfA_sel = pd.concat(sel) if sel else pd.DataFrame()
dfagg_src = pd.concat([df[df.instance != "A"], dfA_sel]) if len(df) else df
agg = []
for keys, g in dfagg_src.groupby(["task", "instance", "synth", "prompt"], dropna=False):
    accs = g["all_acc"].fillna(0).tolist() if keys[1] != "A" else g["test_acc"].tolist()
    mu, sd, ci = t_ci(accs); n = len(g)
    k = int(g["success"].fillna(False).sum()) if "success" in g and keys[1] == "B" else int(sum(a >= SUCCESS_THRESH for a in accs))
    lo, hi = wilson(k, n)
    row = {"task": keys[0], "instance": keys[1], "synth": keys[2], "prompt": keys[3], "n": n, "acc_mean": mu, "acc_std": sd, "acc_ci95": ci,
           "success_rate": k / n if n else np.nan, "success_ci_lo": lo, "success_ci_hi": hi,
           "valid_code_rate": float(g["ok"].mean()) if "ok" in g else np.nan}
    if keys[1] == "A": row.update(lr=g["lr"].iloc[0], steps=g["steps"].mean(), compute_s=g["compute_s"].mean())
    for c in ["acc_noise0.1", "acc_noise0.3", "acc_noise0.5", "mean_margin", "min_margin", "W1_eff_rank", "W1_sparsity",
              "fourier_top10_energy", "in_tok", "out_tok", "cost_usd", "synth_wall_s"]:
        if c in g: row[c] = pd.to_numeric(g[c], errors="coerce").mean()
    agg.append(row)
dfagg = pd.DataFrame(agg); dfagg.to_csv(f"{OUT_DIR}/comparison_table.csv", index=False)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 250); dfagg

In [ ]:
# ---- Plots ----
try:
    dfc = pd.DataFrame(A_CURVES)
    tasks_all = sorted(set(dfagg.task)) if len(dfagg) else []
    if len(dfc):
        fig, axs = plt.subplots(1, len(tasks_all), figsize=(5 * len(tasks_all), 4), squeeze=False)
        for ax, t in zip(axs[0], tasks_all):
            g = dfc[dfc.task == t]
            for (o, lr), gg in g.groupby(["opt", "lr"]):
                mm = gg.groupby("step")[["val_acc", "train_compute_s"]].mean(); ax.plot(mm.index, mm.val_acc, "o-", ms=3, label=f"A {o} lr={lr}")
            for _, r in dfagg[(dfagg.task == t) & (dfagg.instance.isin(["B", "B+ridge"]))].iterrows():
                if r.prompt in ("full", "-"): ax.axhline(r.acc_mean, ls=":", lw=1, label=f"{r.instance} {r.synth} {r.prompt}")
            ax.set_xscale("log"); ax.set_title(t); ax.set_xlabel("step"); ax.set_ylabel("accuracy"); ax.legend(fontsize=5)
        plt.tight_layout(); plt.savefig(f"{OUT_DIR}/accuracy_vs_steps.png", dpi=150); plt.show()
    b = dfagg[dfagg.instance == "B"]
    if len(b):
        fig, axs = plt.subplots(1, len(set(b.task)), figsize=(5 * len(set(b.task)), 4), squeeze=False)
        for ax, (t, g) in zip(axs[0], b.groupby("task")):
            piv = g.pivot_table(index="prompt", columns="synth", values="success_rate")
            piv.plot.bar(ax=ax); ax.set_title(f"B success rate: {t}"); ax.set_ylim(0, 1.05)
        plt.tight_layout(); plt.savefig(f"{OUT_DIR}/B_success_by_prompt.png", dpi=150); plt.show()
    fig, axs = plt.subplots(1, max(1, len(tasks_all)), figsize=(5 * max(1, len(tasks_all)), 4), squeeze=False)
    for ax, t in zip(axs[0], tasks_all):
        for _, r in dfagg[dfagg.task == t].iterrows():
            if r.instance == "B" and r.prompt not in ("full",): continue
            ys = [r.acc_mean] + [r.get(f"acc_noise{n}", np.nan) for n in NOISES]; ax.plot([0] + NOISES, ys, "o-", ms=3, label=f"{r.instance} {r.synth}")
        ax.set_title(f"robustness: {t}"); ax.set_xlabel("input noise std"); ax.legend(fontsize=5)
    plt.tight_layout(); plt.savefig(f"{OUT_DIR}/robustness.png", dpi=150); plt.show()
except Exception as e: print("plotting failed:", repr(e)); traceback.print_exc()

In [ ]:
# ---- Final results.json + zip ----
results = {"config": {k: globals()[k] for k in ["QUICK", "MOCK", "RUN_TASKS", "PROMPT_LEVELS", "N_REPEATS_ALGO", "N_REPEATS_LOCAL",
           "N_REPEATS_CIFAR", "SUCCESS_THRESH", "CLAUDE_TEMPERATURE", "CLAUDE_MAX_TOKENS", "PRICE_PER_MTOK", "CLAUDE_BUDGET_USD",
           "LOCAL_MAX_NEW_TOKENS", "LOCAL_TEMPERATURE", "LOCAL_TOP_P", "SEEDS_A", "LR_GRID", "WEIGHT_DECAY", "CHECKPOINTS", "SPLIT",
           "CIFAR_STEPS", "CIFAR_LR_GRID", "P", "H_ALGO", "PAR_N", "PAR_K", "PAR_SUBSET", "G_ROOT"]},
           "synthesizers": SYNTHESIZERS, "synth_status": SYN_STATUS, "llm_usage": USAGE, "claude_spend_usd": CLAUDE_SPENT[0],
           "env": {"torch": torch.__version__, "python": platform.python_version(), "device": DEVICE,
                   "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]},
           "elapsed_s": time.time() - T_START, "comparison": json.loads(dfagg.to_json(orient="records")),
           "per_run": json.loads(pd.DataFrame(PER_RUN).drop(columns=["pred"], errors="ignore").to_json(orient="records")), "C": C_ROWS}
json.dump(results, open(f"{OUT_DIR}/results.json", "w"), indent=1, default=str)
import shutil
_z = shutil.make_archive(os.path.join(tempfile.gettempdir(), "wsp_v2_outputs"), "zip", OUT_DIR)
zip_path = shutil.move(_z, os.path.join(OUT_DIR, "wsp_v2_outputs.zip"))
print("elapsed %.1f min | Claude $%.3f | zip: %s" % ((time.time() - T_START) / 60, CLAUDE_SPENT[0], zip_path))